# ML00 · 机器学习总览

用图与直觉创建机器学习 machine learning 的整体心智模型，先看懂「学习在学什么」，再进入后续技术细节。

## 学习目标
- 能用自己的话说明机器学习 machine learning 是什么，以及它与传统写死规则的差别。
- 能区分监督式 supervised 与非监督式 unsupervised 学习，并判断一个问题属于分类 classification 还是回归 regression。
- 理解特征 feature 与标签 label 的角色，知道数据如何喂给模型。
- 创建「学习就是最小化误差」的直觉，看懂损失 loss 的意义。
- 理解为什么要做训练/验证/测试切分 train/validation/test split，以及它如何防止自欺。
- 能用图说明过拟合 overfitting 与泛化 generalization，知道「背答案」与「真会」的差别。

In [ ]:
# 中文字体设置（本书会画图才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 什么是机器学习

机器学习 machine learning 是「让程序从数据中自己找出规则」的方法，而不是由人把规则一条条写死。

传统的规则式程序 rule-based program 是人先想好判断逻辑（例如「楼高超过 60 公尺就算高楼」），再写成 if 条件。数据驱动 data-driven 的做法则相反：给一批带答案的样本，让模型 model 自己抓出门槛，最后得到一个会做预测 prediction 的函数。

整体画面就是一句话：**给数据，得到一个会预测的函数**。何时用机器学习？当规则太复杂、人难以穷举，或规则会随数据变动时。

In [ ]:
# 概念：规则式程序 vs 数据驱动，对比「人写死门槛」与「模型自己抓门槛」
import numpy as np

# 造一批假楼高数据（公尺）与是否为高楼的标准答案当示范用
heights = np.array([18, 25, 33, 45, 58, 66, 72, 85, 95, 120])  # 每栋楼的高度
is_high = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])             # 1 表示高楼，0 表示非高楼

# 规则式：人事先写死门槛 60 公尺
rule_threshold = 60
rule_pred = (heights > rule_threshold).astype(int)  # 人写的规则直接判断

# 数据驱动：不写死门槛，从数据自己找「最后一栋非高楼」与「第一栋高楼」的中点
last_low = heights[is_high == 0].max()              # 标签为 0 的最高楼
first_high = heights[is_high == 1].min()            # 标签为 1 的最矮楼
learned_threshold = (last_low + first_high) / 2     # 模型从数据长出的门槛
learned_pred = (heights > learned_threshold).astype(int)

print('人写死的门槛：', rule_threshold, '公尺')
print('从数据学到的门槛：', learned_threshold, '公尺')
print('学到的门槛与标准答案是否完全一致：', bool((learned_pred == is_high).all()))

## 学习的两大家族：监督式与非监督式

有没有标准答案（标签 label）决定了学习属于哪个家族。

监督式学习 supervised learning 有标签，模型学「输入到答案」的对应；非监督式学习 unsupervised learning 没有标签，目标是在数据里找出结构。

监督式又依输出型态分两种：

| 任务 | 输出型态 | 例子 |
|---|---|---|
| 回归 regression | 连续数值 | 预测房价 |
| 分类 classification | 类别标签 | 判断住宅 / 商办 |
| 聚类 clustering（非监督式） | 自动分组 | 把街区聚类 |

In [ ]:
# 概念：用同一份小数据对照回归、分类、聚类三种任务
# 造一张假街区数据：面积（平方公尺）、房价（万元）、用途类别
area = np.array([60, 85, 120, 45, 200, 150])         # 特征：面积
price = np.array([900, 1200, 1700, 700, 2600, 2100]) # 回归的标签：连续房价
usage = np.array(['住宅', '住宅', '商办', '住宅', '商办', '商办'])  # 分类的标签：类别

# 回归：输出是连续数值，这里看房价的范围
print('回归任务（预测房价）标签范围：', price.min(), '到', price.max(), '万元')

# 分类：输出是有限类别
print('分类任务（住宅/商办）类别有：', set(usage))

# 聚类：没有用到任何标签，只看特征就把街区依面积分成两组
# 技巧：最简单的聚类可用一个门槛把样本切两群，体会「无标签找结构」
group = np.where(area < area.mean(), '小面积群', '大面积群')
for a, g in zip(area, group):
    print(f'面积 {a:>3} -> {g}')

## 数据的语言：特征与标签

模型只看得懂数字化的数据，因此要先学会用数据表的语言描述问题。

- 样本 sample：一列就是一个观测对象（例如一栋房屋）。
- 特征 feature：拿来当输入的那些栏（楼高、面积、屋龄）。
- 特征矢量 feature vector：一个样本所有特征组成的一排数字。
- 标签 label：要预测的那一栏（房价）。

理解「一列是一个样本、栏是特征、要预测的那一栏是标签」，是后续所有内容的基础。

In [ ]:
# 概念：特征矩阵 X 与标签矢量 y 的摆放方式
# 造一张含楼高、面积、屋龄的小表，最后一栏房价当标签
feature_names = ['楼高(m)', '面积(m2)', '屋龄(年)']
X = np.array([
    [18,  60,  5],
    [33,  85, 12],
    [66, 120,  2],
    [95, 200, 20],
])                      # 每一列是一个样本，每一栏是一个特征
y = np.array([900, 1200, 1700, 2600])  # 标签：房价（万元），与 X 的列一一对应

print('特征字段：', feature_names)
print('特征矩阵 X 形状（样本数, 特征数）：', X.shape)
print('标签矢量 y 形状（样本数,）：', y.shape)
print('第 0 个样本的特征矢量：', X[0], '对应标签：', y[0])

## 学习就是最小化误差：损失的直觉

学习的核心动作是「先猜，再看错多少，再调整让错变小」。

损失 loss 是一个衡量「预测与真值差多少」的数字；误差 error 越大，损失越大。学习就是不断调整模型，让损失最小化 minimization。

一个最常见的损失是均方误差 mean squared error：把每个样本的误差平方后取平均。直觉上，一条直线越贴近数据点，总误差越小，损失也越小。

In [ ]:
# 概念：用均方误差比较几条候选直线谁更贴近数据
# 造一组面积 -> 房价的假散点（带一点杂讯）
rng = np.random.default_rng(0)                 # 固定乱数种子，结果可重现
area = np.linspace(50, 200, 20)                # 面积从 50 到 200 取 20 点
price = 10 * area + 200 + rng.normal(0, 80, 20)  # 真实关系加上杂讯当作观测房价

def mse(slope, intercept):
    pred = slope * area + intercept            # 用直线做预测
    return np.mean((price - pred) ** 2)        # 均方误差：误差平方后取平均

# 三条候选直线（斜率, 截距），其中一条接近真实关系
candidates = [(6, 400), (10, 200), (14, 0)]
for slope, intercept in candidates:
    print(f'直线 y={slope}x+{intercept:<4} 的损失(MSE)= {mse(slope, intercept):.1f}')

plt.figure(figsize=(6, 4))
plt.scatter(area, price, color='gray', label='数据点')
for slope, intercept in candidates:
    plt.plot(area, slope * area + intercept, label=f'y={slope}x+{intercept}')
plt.xlabel('面积(m2)'); plt.ylabel('房价(万元)')
plt.title('不同直线的拟合与损失')
plt.legend(); plt.show()

## 不要自欺：数据切分与评估

在看过的数据上表现好不算数，因为模型可能只是把答案背起来。

标准做法是把数据切成三份：

| 子集 | 用途 |
|---|---|
| 训练集 training set | 拿来学、调整模型 |
| 验证集 validation set | 调参数、选模型时偷看的成绩 |
| 测试集 test set | 全程不碰，最后一次验收 |

用没看过的数据评估，才知道模型能不能泛化 generalization，避免「考古题全对、真考试挂掉」。

In [ ]:
# 概念：把数据随机切成训练/验证/测试三份
n = 20
samples = np.arange(n)                          # 假设有 20 个样本，用编号代表
rng = np.random.default_rng(42)
shuffled = rng.permutation(samples)             # 先打乱，避免原始顺序带来偏差

# 注意：切分比例常见为 70/15/15，依数据量调整
n_train = int(n * 0.7)
n_val = int(n * 0.15)
train_idx = shuffled[:n_train]                  # 前 70% 拿来训练
val_idx = shuffled[n_train:n_train + n_val]     # 接着 15% 拿来验证
test_idx = shuffled[n_train + n_val:]           # 剩下留作测试，全程不碰

print('训练集样本数：', len(train_idx))
print('验证集样本数：', len(val_idx))
print('测试集样本数：', len(test_idx))
print('三份是否互不重叠：', len(set(train_idx) & set(test_idx)) == 0)

## 过拟合与泛化：背答案 vs 真会

把训练数据背得太细，反而会把杂讯也学进去，换新数据就失准。

- 欠拟合 underfitting：模型太简单，连训练数据都学不好。
- 过拟合 overfitting：模型太弯，训练误差很小但测试误差大。
- 泛化 generalization：训练与测试表现都好，才是「真会」。

判断关键是看模型复杂度 model complexity 与两组误差的落差：训练好、测试差，就是背答案。

In [ ]:
# 概念：用不同次数的多项式拟合，对照欠拟合、适中、过拟合
rng = np.random.default_rng(1)
x = np.linspace(0, 1, 15)                        # 自造带杂讯的点
y_true = np.sin(2 * np.pi * x)                   # 背后真实规律
y = y_true + rng.normal(0, 0.25, x.shape)        # 加杂讯得到观测值

xs = np.linspace(0, 1, 200)                       # 画平滑曲线用的密集点
degrees = {'欠拟合(次数1)': 1, '适中(次数3)': 3, '过拟合(次数12)': 12}

plt.figure(figsize=(6, 4))
plt.scatter(x, y, color='gray', label='数据点')
for label, deg in degrees.items():
    coef = np.polyfit(x, y, deg)                 # 用多项式拟合，次数越高越能弯曲
    train_err = np.mean((np.polyval(coef, x) - y) ** 2)
    plt.plot(xs, np.polyval(coef, xs), label=f'{label} 训练误差={train_err:.3f}')
plt.ylim(-2, 2)
plt.xlabel('x'); plt.ylabel('y')
plt.title('欠拟合 / 适中 / 过拟合')
plt.legend(fontsize=8); plt.show()

## 练习

以下三题由浅入深，皆为复合型但可快速完成。数据请自己用 numpy / list 造，不要引用任何外部文件。

In [ ]:
# TODO 1 ·（简单）帮房屋数据粘贴正确标签（集成：特征与标签 + 监督式分类与回归）
#   情境：你拿到一小批自造的房屋数据，每列有楼高、面积、屋龄，以及房价与「住宅/商办」两个字段。
#   要求：
#     1. 用 numpy / list 自造这份小数据，并指出哪些栏是特征 feature、哪一栏会被当成标签 label。
#     2. 判断「预测房价」与「判断住宅或商办」各属于回归 regression 还是分类 classification，并印出说明依据。
#   验收：应该看到一份清楚标示特征与标签的对照，以及两个任务各自被正确归类为回归或分类。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）用最小误差挑一条房价线并诚实评估（集成：损失直觉 + 数据切分 + 监督式回归）
#   情境：用自造的「面积 -> 房价」散点，仿真一个最简单的回归问题。
#   要求：
#     1. 先把数据切成训练/验证/测试 train/validation/test split。
#     2. 在训练数据上比较两三条候选直线，依损失 loss（误差大小）选出较好的一条。
#     3. 用没看过的测试数据计算被选中直线的损失，判断是否仍然准。
#   验收：应该看到被选中的直线在训练集上误差最小，并能说出它在测试集上是否一样表现良好。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）找出背答案的那条曲线（集成：过拟合与泛化 + 损失 + 数据切分 + 模型复杂度）
#   情境：用自造、带一点杂讯的容积率与楼高关系数据，准备三种复杂度不同的拟合线（很直、适中、很弯）。
#   要求：
#     1. 切出训练集与测试集，对三种拟合线分别计算它们在两组数据上的损失 loss。
#     2. 指出哪一条是过拟合 overfitting、哪一条欠拟合、哪一条泛化 generalization 最好，并说明从两组误差的落差如何判断。
#     3. 简短印出：若想避免过拟合，可以朝哪个方向调整（如降低复杂度或增加数据）。
#   验收：应该看到「训练误差很小但测试误差明显变大」的那条被指认为过拟合，泛化最佳的那条在两组数据上表现接近。
# TODO: 在这里完成

## 小结

- 机器学习 machine learning 是从数据中长出规则，核心画面是「给数据，得到一个会预测的函数」。
- 有没有标签 label 决定学习家族：监督式 supervised 分为回归 regression（连续数值）与分类 classification（类别），非监督式 unsupervised 则做聚类 clustering 找结构。
- 数据用「列是样本 sample、栏是特征 feature、要预测的那栏是标签 label」的语言描述。
- 学习就是最小化损失 loss：先猜、看误差、再调整，让预测越来越贴近真值。
- 用训练/验证/测试切分 train/validation/test split 在没看过的数据上评估，才能避免自欺。
- 真会等于能泛化 generalization；训练好但测试差就是过拟合 overfitting，背了答案却没学到规律。